In [ ]:
import json  # Parse BigQuery's JSON response into Python records.
from io import StringIO  # Let pandas read BigQuery command output held in memory.
import os  # Set a temporary Matplotlib configuration directory below.
import subprocess  # Run authenticated BigQuery CLI commands from Python.
import tempfile  # Find a safe temporary directory for Matplotlib files.
from pathlib import Path  # Build filesystem paths in an OS-independent way.

os.environ.setdefault(
    "MPLCONFIGDIR",
    str(Path(tempfile.gettempdir()) / "amazon_books_matplotlib"),
)

import matplotlib.pyplot as plt  # Create plots later in the notebook.
import numpy as np  # Numerical arrays and calculations.
import pandas as pd  # Load, inspect, and transform table-shaped data.
from IPython.display import display

PROJECT = "wagon-bootcamp-501605"
DATASET = "amazon_books_2023"
TABLE = "business_money_usable_tocs_9034"
LOCATION = "asia-northeast1"
RANDOM_STATE = 42
FULL_TABLE = f"{PROJECT}.{DATASET}.{TABLE}"
print(FULL_TABLE)

# Prepare the usable TOCs

The BigQuery table is already a curated source: every record has an exact-ISBN Open Library match and at least three extracted TOC entries. We keep the raw fields unchanged and build a separate, model-ready table.

Following the data-preparation lecture, we audit duplicates and missing data before transforming text. The unit of analysis is `parent_asin`, not ISBN: 39 canonical ISBNs occur in more than one Amazon record.

In [ ]:
# Load every source column first. JSON preserves nested and repeated fields
# that BigQuery cannot serialize in CSV.
query = f"""
SELECT *
FROM `{FULL_TABLE}`
ORDER BY parent_asin
"""

command = [
    "bq",
    f"--location={LOCATION}",
    "query",
    "--use_legacy_sql=false",
    "--format=json",
    "--max_rows=10000",
    query,
]
print("1/7 -- Full-table BigQuery command is ready.")

In [ ]:
# Run the full-table query. Its JSON output can preserve nested columns. (50 seconds)
result = subprocess.run(command, capture_output=True, text=True, check=True)
print(f"2/7 -- Downloaded {len(result.stdout):,} characters of JSON.")

In [ ]:
# Convert the JSON response into one pandas row per book. (9,034 rows x 38 columns.)
source_books = pd.DataFrame(json.loads(result.stdout))
print(f"3/7 -- Created source_books: {source_books.shape[0]:,} rows x {source_books.shape[1]:,} columns.")

In [ ]:
# Store identifier-like values as text so leading zeros and ISBN-10's final X are never lost.
identifier_columns = [
    "parent_asin", "amazon_isbn_10", "amazon_isbn_13",
    "canonical_isbn_13", "ol_edition_key",
]
for column in identifier_columns:
    source_books[column] = source_books[column].astype("string")
print("4/7 -- Identifier dtypes:")
print(source_books[identifier_columns].dtypes)

In [ ]:
# CHECKPOINT -- before 5/7: we have preserved the original book-level table.
print("Before TOC flattening")
print(f"- source_books contains {len(source_books):,} books and {len(source_books.columns):,} original columns.")
print("- Each row is one Amazon book; nested fields such as toc_entries are still intact.")
print("- Next we create a separate entry-level view only to validate and clean TOC text.")
display(source_books[["parent_asin", "title", "toc_entry_count", "toc_entries"]].head(3))

In [ ]:
# source_books has one row per book and preserves every original column.
# This separate query does not append to or replace source_books. It creates
# toc_items: one row per nested TOC entry, which lets us check entry order,
# blanks, and duplicates before joining only cleaned toc_text back to books.
#
# Before flattening, one book stores its TOC as a list inside one cell:
# parent_asin = "000216132X"
# toc_entries = [{"sequence": 1, "text": "v. 1. The structures..."},
#                {"sequence": 2, "text": "v. 2. The wheels..."},
#                {"sequence": 3, "text": "v. 3. The perspective..."}]
# After flattening, those become three rows with parent_asin, toc_sequence,
# and toc_text_raw. The displays below use a real book from this table.
example_book = source_books.iloc[0]
example_parent_asin = example_book["parent_asin"]
print(f"Nested TOC -- real book: {example_parent_asin} | {example_book['title']}")
display(pd.DataFrame(example_book["toc_entries"])[["sequence", "text"]])
toc_query = f"""
SELECT parent_asin, raw_toc_item_count, toc_entry_count,
  entry.sequence AS toc_sequence, entry.text AS toc_text_raw,
  entry.level AS toc_level, entry.page AS toc_page
FROM `{FULL_TABLE}`
CROSS JOIN UNNEST(toc_entries) AS entry
ORDER BY parent_asin, toc_sequence
"""
toc_command = [
    "bq", f"--location={LOCATION}", "query",
    "--use_legacy_sql=false", "--format=csv", "--max_rows=1000000", toc_query,
]
print("5/7 -- Flattened TOC query is ready.")

In [ ]:
# Run the simple flat query and read its CSV response into pandas.
toc_result = subprocess.run(toc_command, capture_output=True, text=True, check=True)
toc_items = pd.read_csv(StringIO(toc_result.stdout), dtype={"parent_asin": "string"})
print(f"6/7 -- Loaded {len(toc_items):,} TOC entries for {toc_items['parent_asin'].nunique():,} books.")

In [ ]:
# Inspect one book without truncating its title or TOC text.
BOOK_TO_INSPECT = "000216132X"
book_record = source_books.loc[source_books["parent_asin"] == BOOK_TO_INSPECT].iloc[0]
book_toc = toc_items.loc[
    toc_items["parent_asin"] == BOOK_TO_INSPECT,
    ["toc_sequence", "toc_text_raw", "toc_level", "toc_page"],
].sort_values("toc_sequence")

print(f"Amazon title: {book_record['title']}")
print(f"Open Library edition: {book_record['ol_edition_key']}")
print(f"Declared TOC entries: {book_record['toc_entry_count']}")
print("\nFull table of contents:")
for entry in book_toc.itertuples(index=False):
    print(f"{entry.toc_sequence}. {entry.toc_text_raw}")

In [ ]:
# CHECKPOINT -- after 5/7 and 6/7: the original table is still separate.
print("After TOC flattening")
print(f"- source_books is unchanged: {len(source_books):,} books x {len(source_books.columns):,} columns.")
print(f"- toc_items is a separate working view: {len(toc_items):,} TOC-entry rows for {toc_items['parent_asin'].nunique():,} books.")
print("- Next we audit TOC entries, clean their text, then join one cleaned toc_text column back to the full book table.")
print(f"Flattened TOC -- same real book: {example_parent_asin}")
display(
    toc_items.loc[toc_items["parent_asin"] == example_parent_asin,
                  ["parent_asin", "toc_sequence", "toc_text_raw"]]
)

In [ ]:
# Inspect three full source records. random_state makes the sample repeatable.
sampled_books = source_books.sample(n=3, random_state=RANDOM_STATE)
for _, record in sampled_books.iterrows():
    print(json.dumps(record.to_dict(), indent=2, ensure_ascii=False, default=str))
    print("\n" + "-" * 100 + "\n")

## 1. Audit duplicates and missing values

Do not remove a book merely because it shares an ISBN or an Open Library edition with another Amazon record. First inspect the fields below; only exact duplicate `parent_asin`/sequence pairs or blank TOC text would be invalid at this grain.

In [ ]:
book_counts = toc_items.groupby("parent_asin").agg(
    observed_entries=("toc_sequence", "size"),
    declared_entries=("toc_entry_count", "first"),
)

quality = pd.Series(
    {
        "TOC-entry rows": len(toc_items),
        "books (parent_asin)": toc_items["parent_asin"].nunique(),
        "duplicate parent_asin / sequence pairs": toc_items.duplicated(["parent_asin", "toc_sequence"]).sum(),
        "blank TOC entries": toc_items["toc_text_raw"].fillna("").str.strip().eq("").sum(),
        "books below the 3-entry rule": (book_counts["declared_entries"] < 3).sum(),
        "books whose observed and declared entry counts differ": (book_counts["observed_entries"] != book_counts["declared_entries"]).sum(),
        "books where raw and normalized TOC counts differ": (
            toc_items.drop_duplicates("parent_asin")["raw_toc_item_count"]
            != toc_items.drop_duplicates("parent_asin")["toc_entry_count"]
        ).sum(),
    },
    name="count",
).to_frame()
display(quality)
display(toc_items.isna().mean().sort_values(ascending=False).rename("missing_share").to_frame())

## 2. Create book-level TOC text (without changing content)

No content transformation is applied yet: TOC numbers, punctuation, capitalization, page numbers, and hierarchy may be meaningful. We validate the raw entries and add one convenient book-level `toc_text` column while keeping every raw field available.

In [ ]:
# 2.1 -- Validate the raw TOC structure before using it.
duplicate_pairs = toc_items.duplicated(["parent_asin", "toc_sequence"]).sum()
blank_raw_entries = toc_items["toc_text_raw"].isna().sum() + toc_items["toc_text_raw"].eq("").sum()
assert duplicate_pairs == 0
assert blank_raw_entries == 0
print(f"2.1 -- Validation passed: {duplicate_pairs} duplicate book/sequence pairs; {blank_raw_entries} blank raw entries.")

In [ ]:
# 2.2 -- Inspect a raw book, then reassemble its ordered raw entries into toc_text.
raw_book = source_books.loc[source_books["parent_asin"] == BOOK_TO_INSPECT].iloc[0]
print("Raw source record (before any content transformation):")
print(f"Title: {raw_book['title']}")
print(f"Amazon ISBN-10: {raw_book['amazon_isbn_10']}")
print(f"Amazon ISBN-13: {raw_book['amazon_isbn_13']}")
print(f"Canonical ISBN-13: {raw_book['canonical_isbn_13']}")
print(f"Parent ASIN: {raw_book['parent_asin']}")
print("\nOriginal nested toc_entries:")
print(json.dumps(raw_book["toc_entries"], indent=2, ensure_ascii=False))

toc_text_by_book = (
    toc_items.sort_values(["parent_asin", "toc_sequence"])
    .groupby("parent_asin")["toc_text_raw"]
    .agg("\n".join)
)
toc_text_by_book.name = "toc_text"
print(f"2.2 -- Created one ordered TOC text value for {len(toc_text_by_book):,} books.")
print(f"\nTOC for {BOOK_TO_INSPECT}:\n{toc_text_by_book.loc[BOOK_TO_INSPECT]}")

## 3. strip(), lower case (when embedding words upcase/lowercase matter)


In [ ]:
# 3 -- Optional variant: trim only the outside whitespace and lowercase TOC text.
# The unchanged books['toc_text'] remains our baseline and is never overwritten.
books = source_books.copy().join(toc_text_by_book, on="parent_asin")
books["toc_text_lower"] = books["toc_text"].str.strip().str.lower()
example = books.loc[books["parent_asin"] == BOOK_TO_INSPECT].iloc[0]
print("3 -- Optional lowercase variant created; baseline toc_text is unchanged.")
print(f"Baseline:  {example['toc_text']!r}")
print(f"Lowercase: {example['toc_text_lower']!r}")

## 4. dealing with numbers, punctuation, and symbols

In [ ]:
# 4 -- Optional variant: remove digits, punctuation, and symbols from a copy.
# This is deliberately not applied to toc_text because TOC markers such as 1. and Part II may matter.
import re
books["toc_text_letters_only"] = (
    books["toc_text"].str.replace(r"[^\w\s]|\d", " ", regex=True).str.replace(r"\s+", " ", regex=True).str.strip()
)
example = books.loc[books["parent_asin"] == BOOK_TO_INSPECT].iloc[0]
print("4 -- Optional letters-only variant created; baseline toc_text is unchanged.")
print(f"Baseline:     {example['toc_text']!r}")
print(f"Letters only: {example['toc_text_letters_only']!r}")

## 5. splitting

In [ ]:
# 5 -- Optional structural view: split the multiline TOC into its original entries.
# This creates a list for analysis; it does not change toc_text.
# pandas has no .str.splitlines(); our book-level toc_text uses newline separators.
books["toc_lines"] = books["toc_text"].str.split("\n")
books["toc_line_count"] = books["toc_lines"].str.len()
example = books.loc[books["parent_asin"] == BOOK_TO_INSPECT].iloc[0]
print(f"5 -- Split TOC into {example['toc_line_count']} lines for {BOOK_TO_INSPECT}.")
for position, line in enumerate(example["toc_lines"], start=1):
    print(f"{position}. {line}")

## 6. tokenizing

In [ ]:
# 6 -- Optional token view: extract word-like tokens into a separate list.
# Most vectorizers/tokenizers do this themselves, so toc_text stays the baseline input.
books["toc_tokens"] = books["toc_text"].str.findall(r"\b\w+\b")
example = books.loc[books["parent_asin"] == BOOK_TO_INSPECT].iloc[0]
print(f"6 -- Extracted {len(example['toc_tokens'])} tokens for {BOOK_TO_INSPECT}.")
print(example["toc_tokens"][:40])

## 7. removing "stopwords"

In [ ]:
# 7 -- Optional token variant: remove English stopwords from a copy of the token list.
# Keep this experimental: words such as 'part' or 'chapter' may matter for TOCs.
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
books["toc_tokens_no_stopwords"] = books["toc_tokens"].map(
    lambda tokens: [token for token in tokens if token.casefold() not in ENGLISH_STOP_WORDS]
)
example = books.loc[books["parent_asin"] == BOOK_TO_INSPECT].iloc[0]
print(f"7 -- Tokens: {len(example['toc_tokens'])}; after stopword removal: {len(example['toc_tokens_no_stopwords'])}.")
print(example["toc_tokens_no_stopwords"][:40])

## 8. lemmatizing

In [ ]:
# 8 -- Optional token variant: apply simple noun lemmatization to a copy.
# This requires nltk in the current notebook kernel; all earlier steps work without it.
try:
    from nltk.stem import WordNetLemmatizer
except ModuleNotFoundError:
    print("8 -- Skipped: nltk is not installed in this notebook kernel.")
    print("Run `%pip install -e .` from the project folder, restart the kernel, then rerun this optional cell.")
else:
    lemmatizer = WordNetLemmatizer()
    try:
        books["toc_tokens_lemmatized"] = books["toc_tokens"].map(
            lambda tokens: [lemmatizer.lemmatize(token.casefold()) for token in tokens]
        )
    except LookupError:
        print("8 -- Skipped: the NLTK WordNet data is not installed.")
        print("Run `import nltk; nltk.download('wordnet')`, then rerun this optional cell.")
    else:
        example = books.loc[books["parent_asin"] == BOOK_TO_INSPECT].iloc[0]
        print("8 -- Original tokens versus optional noun-lemmatized tokens:")
        print(f"Original:    {example['toc_tokens'][:40]}")
        print(f"Lemmatized:  {example['toc_tokens_lemmatized'][:40]}")

In [ ]:
print(f"Intermediate result: {len(books):,} rows x {len(books.columns):,} columns.")
print(f"Columns: {books.columns.tolist()}")

In [ ]:
difficulty_query = """
SELECT *
FROM `wagon-bootcamp-501605.amazon_books_2023.difficulty_classifications_8995`
"""
difficulty_command = [
    "bq", "--location=asia-northeast1", "query",
    "--use_legacy_sql=false", "--format=json", "--max_rows=100000",
    difficulty_query,
]
difficulty_result = subprocess.run(difficulty_command, capture_output=True, text=True, check=True)
difficulty = pd.DataFrame(json.loads(difficulty_result.stdout))
print(f"difficulty_classifications_8995: {difficulty.shape[0]:,} rows x {difficulty.shape[1]:,} columns.")
print(f"Columns: {difficulty.columns.tolist()}")

assert "parent_asin" in difficulty.columns, "No parent_asin found -- check the real join key!"
difficulty["parent_asin"] = difficulty["parent_asin"].astype("string")

In [ ]:
books = books.merge(
    difficulty[["parent_asin", "difficulty_score"]],
    on="parent_asin",
    how="left",
)

print(f"books: {len(books):,} rows x {len(books.columns):,} columns.")
print(f"Missing difficulty_score: {books['difficulty_score'].isna().sum():,}")
print(books["difficulty_score"].value_counts(dropna=False).sort_index())
display(books[["parent_asin", "title", "toc_text", "difficulty_score"]].head(3))

In [ ]:
def extract_main_image_url(raw_json):
    variants = json.loads(raw_json) if isinstance(raw_json, str) else raw_json
    if not variants:
        return None
    for variant in variants:
        if variant.get("variant") == "MAIN" and variant.get("large"):
            return variant["large"]
    return variants[0].get("large")


def extract_leaf_category(raw_categories):
    categories_list = json.loads(raw_categories) if isinstance(raw_categories, str) else raw_categories
    if isinstance(categories_list, list) and len(categories_list) > 0:
        return categories_list[-1]
    return None


books["cover_image_url"] = books["images_json"].map(extract_main_image_url)
books["category"] = books["categories"].map(extract_leaf_category)
books["ISBN_10"] = books["amazon_isbn_10"]

print(f"Missing cover_image_url: {books['cover_image_url'].isna().sum():,}")
print(f"Missing category: {books['category'].isna().sum():,}")
display(books[["parent_asin", "title", "cover_image_url", "category", "ISBN_10", "difficulty_score"]].head(3))

In [ ]:
import re
import subprocess
import sys

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Make sure nltk is installed in THIS kernel (the one running this notebook),
# instead of asking for a manual pip install in a separate step.
try:
    import nltk
except ModuleNotFoundError:
    print("nltk is not installed in this kernel -- installing it now...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nltk"], check=True)
    import nltk

from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()
try:
    lemmatizer.lemmatize("test")
except LookupError:
    print("Downloading NLTK wordnet data (one-time)...")
    nltk.download("wordnet")
    nltk.download("omw-1.4")
    lemmatizer.lemmatize("test")  # confirm it works now

LEMMATIZE_AVAILABLE = True
print("Lemmatizer ready:", lemmatizer.lemmatize("books"))


def process_toc_text(text):
    if not isinstance(text, str) or not text:
        return text
    cleaned = text.strip().lower()
    cleaned = re.sub(r"[^\w\s]|\d", " ", cleaned)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    tokens = re.findall(r"\b\w+\b", cleaned)
    tokens = [token for token in tokens if token not in ENGLISH_STOP_WORDS]
    if LEMMATIZE_AVAILABLE:
        tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return " ".join(tokens)


books["toc_text_processed"] = books["toc_text"].map(process_toc_text)

example = books.loc[books["parent_asin"] == BOOK_TO_INSPECT].iloc[0]
print("Baseline:  ", example["toc_text"])
print("Processed: ", example["toc_text_processed"])

In [ ]:
final_columns = [
    "parent_asin", "title", "ISBN_10", "cover_image_url", "category",
    "toc_entries", "toc_text", "toc_text_processed", "difficulty_score",
]
books_final = books[final_columns].copy()

print(f"books_final: {len(books_final):,} rows x {len(books_final.columns):,} columns.")
display(books_final.head(3))

# Optional: export as CSV, e.g. for model.py
# books_final.to_csv("exercise_data.csv", index=False)